# Per-Item Model Comparison — All 4 Categories Combined (Unified Classifier)

Extends the Locomotives-only per-item benchmark to **all 4 categories at
once** — Locomotives, Passenger Train Cars, Freight Cars, Automobiles —
trained as a **single unified classifier** across every `products_id` in
the whole catalog, not 4 separate per-category models.

**Why unified, not 4 separate models:** this matches how the feature
actually gets used. A visitor photographs an item with no prior knowledge
of which category it's in — the system needs to search the *entire*
catalog at once, not first guess 'is this a locomotive or a freight car?'
A single model across all 838 items is the realistic architecture to test.

**Same leakage-safe approach as the Locomotives-only run:** each item's
one real photo is held out as the test image; only synthetic augmented
variants are used for train/validation.

**New in this version:** a per-category accuracy breakdown, so you can see
whether performance holds up evenly across categories or whether a
thinner category (Automobiles has only 21 items) drags results down.

**Models:** EfficientNet-B0, MobileNetV2 (the two CNN finalists), and CLIP
ViT-B/32 (linear probe).

**Dataset path:**
```
/kaggle/input/datasets/datascientist97/locomotive-collection-images/conductor_dataset
```

## 0. Setup

In [ ]:
!pip install -q timm
# Deliberately NOT using --upgrade and NOT reinstalling torch/torchvision —
# Kaggle's preinstalled versions are matched to the assigned GPU.

In [ ]:
import os
import copy
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models as tvmodels

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, log_loss,
    top_k_accuracy_score, roc_auc_score, classification_report
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

FORCE_CPU = False  # set True if you hit 'no kernel image available' CUDA errors
DEVICE = torch.device('cpu') if FORCE_CPU else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

if torch.cuda.is_available() and not FORCE_CPU:
    try:
        test_t = torch.randn(2, 2).cuda()
        _ = test_t @ test_t
        print('✅ GPU tensor op succeeded.')
    except Exception as e:
        print('❌ GPU tensor op failed — set FORCE_CPU = True above, or switch',
              'Accelerator type in Kaggle settings (e.g. P100 -> T4).')
        print('Error:', e)

In [ ]:
DATASET_ROOT = "/kaggle/input/datasets/datascientist97/locomotive-collection-images/conductor_dataset"
CATEGORIES = ["Locomotives", "Passenger Train Cars", "Freight Cars", "Automobiles"]

IMG_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 25
EARLY_STOPPING_PATIENCE = 6
LEARNING_RATE = 1e-3

for cat in CATEGORIES:
    path = os.path.join(DATASET_ROOT, cat)
    print(f"{cat:24s} -> {'OK' if os.path.isdir(path) else 'MISSING'}")

## 1. Build the Leakage-Safe Split — All 4 Categories Combined

Same rule as before, applied across every category folder: the file
**without** `_aug_` in its name is the real photo → test. The `_aug_`
files split between train and validation. Classes with no real photo or
too few images are skipped and logged, tagged with which category they
came from.

In [ ]:
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
MIN_TRAIN_IMAGES = 2

records = []
skipped_classes = []

for category in CATEGORIES:
    cat_path = os.path.join(DATASET_ROOT, category)
    if not os.path.isdir(cat_path):
        continue

    for products_id in sorted(os.listdir(cat_path)):
        folder = os.path.join(cat_path, products_id)
        if not os.path.isdir(folder):
            continue

        real_files, aug_files = [], []
        for fname in sorted(os.listdir(folder)):
            ext = os.path.splitext(fname)[1].lower()
            if ext not in VALID_EXTS:
                continue
            (aug_files if '_aug_' in fname else real_files).append(fname)

        if not real_files:
            skipped_classes.append((category, products_id, 'no original real photo found'))
            continue
        if len(aug_files) < MIN_TRAIN_IMAGES:
            skipped_classes.append((category, products_id, f'only {len(aug_files)} non-test image(s)'))
            continue

        test_file = real_files[0]
        records.append({'category': category, 'products_id': products_id, 'filename': test_file,
                         'path': os.path.join(folder, test_file), 'split': 'test'})

        random.shuffle(aug_files)
        val_file = aug_files[-1]
        train_files = aug_files[:-1]

        records.append({'category': category, 'products_id': products_id, 'filename': val_file,
                         'path': os.path.join(folder, val_file), 'split': 'val'})
        for f in train_files:
            records.append({'category': category, 'products_id': products_id, 'filename': f,
                             'path': os.path.join(folder, f), 'split': 'train'})

split_df = pd.DataFrame(records)
print(f"Usable items (across all categories): {split_df['products_id'].nunique()}")
print(f"Skipped items: {len(skipped_classes)}")
print()
print('Usable items per category:')
print(split_df.drop_duplicates('products_id').groupby('category').size().reindex(CATEGORIES))
print()
print('Split counts:')
print(split_df['split'].value_counts())

In [ ]:
# IMPORTANT: products_id is assumed globally unique across the whole Zen Cart
# products table (it's the primary key), so it's safe to use as a single
# unified label space spanning all 4 categories — verify that assumption here.
dupe_check = split_df.groupby('products_id')['category'].nunique()
cross_category_dupes = dupe_check[dupe_check > 1]
if len(cross_category_dupes) > 0:
    print(f"⚠️ WARNING: {len(cross_category_dupes)} products_id value(s) appear under "
          f"more than one category — this breaks the unified label assumption:")
    print(cross_category_dupes)
else:
    print('✅ Confirmed: every products_id belongs to exactly one category. Safe to use as a unified label space.')

In [ ]:
NUM_CLASSES = split_df['products_id'].nunique()
CLASS_NAMES = sorted(split_df['products_id'].unique())
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
idx_to_category = split_df.drop_duplicates('products_id').set_index('products_id')['category'].to_dict()

train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df = split_df[split_df['split'] == 'test'].reset_index(drop=True)

print(f"Total classes (unified across 4 categories): {NUM_CLASSES}")
print(f"Train images: {len(train_df)} | Val images: {len(val_df)} | Test images: {len(test_df)}")
print()
print('Test set size per category (should roughly match usable-items-per-category above):')
print(test_df['category'].value_counts().reindex(CATEGORIES))

## 2. Datasets, Transforms & DataLoaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(4),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CatalogDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        img = self.transform(img)
        label = class_to_idx[row['products_id']]
        return img, label

train_loader = DataLoader(CatalogDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(CatalogDataset(val_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(CatalogDataset(test_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 3. Model Builders — EfficientNet-B0 & MobileNetV2

In [ ]:
def build_efficientnet_b0(num_classes, pretrained=True):
    weights = tvmodels.EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = tvmodels.efficientnet_b0(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

def build_mobilenet_v2(num_classes, pretrained=True):
    weights = tvmodels.MobileNet_V2_Weights.DEFAULT if pretrained else None
    model = tvmodels.mobilenet_v2(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

MODEL_BUILDERS = {
    'EfficientNet-B0': build_efficientnet_b0,
    'MobileNetV2': build_mobilenet_v2,
}

## 4. Training Loop — Early Stopping & Best-Checkpoint Restore

Identical logic to the Locomotives-only notebook, just training on the
full 838-item unified label space this time.

In [ ]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def model_size_mb(model):
    tmp_path = '_tmp_size_check.pt'
    torch.save(model.state_dict(), tmp_path)
    size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
    os.remove(tmp_path)
    return size_mb

def measure_inference_speed(model, loader, n_batches=5):
    model.eval()
    times = []
    with torch.no_grad():
        for i, (images, _) in enumerate(loader):
            if i >= n_batches:
                break
            images = images.to(DEVICE)
            start = time.time()
            _ = model(images)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            times.append((time.time() - start) / images.size(0))
    return float(np.mean(times)) * 1000 if times else float('nan')

In [ ]:
def train_and_evaluate(model_name, builder_fn):
    print(f"\n{'='*60}\nTraining: {model_name}\n{'='*60}")

    model = builder_fn(NUM_CLASSES, pretrained=True).to(DEVICE)
    total_params, trainable_params = count_params(model)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}
    best_val_acc = -1.0
    best_epoch = -1
    best_state = None
    epochs_without_improvement = 0

    start_time = time.time()
    for epoch in range(MAX_EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        model.eval()
        val_loss_total, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss_total += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += images.size(0)
        val_loss, val_acc = val_loss_total / val_total, val_correct / val_total

        history['train_acc'].append(train_acc)
        history['train_loss'].append(train_loss)
        history['val_acc'].append(val_acc)
        history['val_loss'].append(val_loss)

        improved = val_acc > best_val_acc
        print(f"  Epoch {epoch+1}/{MAX_EPOCHS} — train_acc: {train_acc:.3f} train_loss: {train_loss:.3f} "
              f"| val_acc: {val_acc:.3f} val_loss: {val_loss:.3f}{' *' if improved else ''}")

        if improved:
            best_val_acc, best_epoch = val_acc, epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"  ⏹ Early stopping — no val improvement for {EARLY_STOPPING_PATIENCE} epochs. "
                      f"Best was epoch {best_epoch} (val_acc {best_val_acc:.3f}).")
                break

    train_time_sec = time.time() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"  Restored best checkpoint from epoch {best_epoch}.")

    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    test_loss_total = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images_dev, labels_dev = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images_dev)
            loss = criterion(outputs, labels_dev)
            test_loss_total += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    test_loss = test_loss_total / len(all_labels)
    test_acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0)

    all_probs_arr = np.array(all_probs)
    k = min(3, NUM_CLASSES)
    top3_acc = top_k_accuracy_score(all_labels, all_probs_arr, k=k, labels=list(range(NUM_CLASSES)))
    try:
        roc_auc = roc_auc_score(all_labels, all_probs_arr, multi_class='ovr', average='macro')
    except Exception:
        roc_auc = float('nan')

    size_mb = model_size_mb(model)
    inference_ms = measure_inference_speed(model, test_loader)

    per_class_report = classification_report(
        all_labels, all_preds, target_names=[str(c) for c in CLASS_NAMES],
        output_dict=True, zero_division=0)

    # Per-category breakdown — the key addition for this multi-category version
    test_df_with_preds = test_df.copy()
    test_df_with_preds['correct'] = [
        (all_preds[i] == all_labels[i]) for i in range(len(all_labels))
    ]
    per_category_acc = test_df_with_preds.groupby('category')['correct'].mean().reindex(CATEGORIES)

    result = {
        'model_name': model_name, 'total_params': total_params, 'trainable_params': trainable_params,
        'model_size_mb': size_mb, 'train_time_sec': train_time_sec, 'inference_ms_per_image': inference_ms,
        'best_epoch': best_epoch, 'epochs_run': len(history['train_acc']),
        'train_acc': history['train_acc'][best_epoch - 1] if best_epoch > 0 else history['train_acc'][-1],
        'train_loss': history['train_loss'][best_epoch - 1] if best_epoch > 0 else history['train_loss'][-1],
        'val_acc': best_val_acc,
        'val_loss': history['val_loss'][best_epoch - 1] if best_epoch > 0 else history['val_loss'][-1],
        'test_acc': test_acc, 'test_loss': test_loss,
        'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1,
        'top3_acc': top3_acc, 'roc_auc_macro': roc_auc,
    }

    return result, history, per_class_report, per_category_acc

## 5. Train Both CNN Models

In [ ]:
all_results = []
all_histories = {}
all_per_class_reports = {}
all_per_category_acc = {}

for model_name, builder_fn in MODEL_BUILDERS.items():
    result, history, per_class_report, per_category_acc = train_and_evaluate(model_name, builder_fn)
    all_results.append(result)
    all_histories[model_name] = history
    all_per_class_reports[model_name] = per_class_report
    all_per_category_acc[model_name] = per_category_acc
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

print('\n✅ Both CNN models trained.')

## 6. CLIP (ViT-B/32) — Frozen Embeddings + Linear Probe

Same approach as before, now classifying across all 838 unified classes.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model_name = 'openai/clip-vit-base-patch32'
clip_model = CLIPModel.from_pretrained(clip_model_name).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model.eval()

def get_pooled_features(output):
    if isinstance(output, torch.Tensor):
        return output
    if hasattr(output, 'image_embeds') and output.image_embeds is not None:
        return output.image_embeds
    if hasattr(output, 'pooler_output') and output.pooler_output is not None:
        return output.pooler_output
    if hasattr(output, 'last_hidden_state'):
        return output.last_hidden_state.mean(dim=1)
    raise TypeError(f"Unrecognized CLIP output type: {type(output)}")

def extract_clip_embeddings(df):
    embeddings, labels = [], []
    batch_size = 32
    paths = df['path'].tolist()
    pids = df['products_id'].tolist()
    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i+batch_size]
        batch_pids = pids[i:i+batch_size]
        images = [Image.open(p).convert('RGB') for p in batch_paths]
        inputs = clip_processor(images=images, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            raw_output = clip_model.get_image_features(**inputs)
            feats = get_pooled_features(raw_output)
        embeddings.extend(feats.cpu().numpy())
        labels.extend([class_to_idx[p] for p in batch_pids])
    return np.array(embeddings), np.array(labels)

In [ ]:
clip_start = time.time()
print('Extracting CLIP embeddings for train/val/test...')
X_train, y_train = extract_clip_embeddings(train_df)
X_val, y_val = extract_clip_embeddings(val_df)
X_test, y_test = extract_clip_embeddings(test_df)
embedding_time = time.time() - clip_start
print(f"Embedding extraction took {embedding_time:.1f}s for "
      f"{len(X_train)+len(X_val)+len(X_test)} images total.")

In [ ]:
clip_head = LogisticRegression(max_iter=3000, multi_class='auto')

head_start = time.time()
clip_head.fit(X_train, y_train)
clip_train_time = time.time() - head_start + embedding_time

test_preds = clip_head.predict(X_test)
test_probs = clip_head.predict_proba(X_test)

clip_test_acc = accuracy_score(y_test, test_preds)
clip_test_loss = log_loss(y_test, test_probs, labels=list(range(NUM_CLASSES)))
clip_precision, clip_recall, clip_f1, _ = precision_recall_fscore_support(
    y_test, test_preds, average='macro', zero_division=0)
clip_top3 = top_k_accuracy_score(y_test, test_probs, k=min(3, NUM_CLASSES), labels=list(range(NUM_CLASSES)))
try:
    clip_roc_auc = roc_auc_score(y_test, test_probs, multi_class='ovr', average='macro')
except Exception:
    clip_roc_auc = float('nan')

val_preds = clip_head.predict(X_val)
clip_val_acc = accuracy_score(y_val, val_preds)
train_preds = clip_head.predict(X_train)
clip_train_acc = accuracy_score(y_train, train_preds)

sample_imgs = [Image.open(p).convert('RGB') for p in test_df['path'].tolist()[:16]]
inf_start = time.time()
inputs = clip_processor(images=sample_imgs, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    raw_output = clip_model.get_image_features(**inputs)
    feats = get_pooled_features(raw_output)
_ = clip_head.predict(feats.cpu().numpy())
clip_inference_ms = (time.time() - inf_start) / len(sample_imgs) * 1000

# Per-category breakdown for CLIP too
test_df_clip = test_df.copy()
test_df_clip['correct'] = (test_preds == y_test)
clip_per_category_acc = test_df_clip.groupby('category')['correct'].mean().reindex(CATEGORIES)
all_per_category_acc['CLIP ViT-B/32 (linear probe)'] = clip_per_category_acc

clip_result = {
    'model_name': 'CLIP ViT-B/32 (linear probe)',
    'total_params': sum(p.numel() for p in clip_model.parameters()),
    'trainable_params': X_train.shape[1] * NUM_CLASSES,
    'model_size_mb': model_size_mb(clip_model),
    'train_time_sec': clip_train_time,
    'inference_ms_per_image': clip_inference_ms,
    'best_epoch': 1, 'epochs_run': 1,
    'train_acc': clip_train_acc, 'train_loss': float('nan'),
    'val_acc': clip_val_acc, 'val_loss': float('nan'),
    'test_acc': clip_test_acc, 'test_loss': clip_test_loss,
    'precision_macro': clip_precision, 'recall_macro': clip_recall, 'f1_macro': clip_f1,
    'top3_acc': clip_top3, 'roc_auc_macro': clip_roc_auc,
}
all_results.append(clip_result)
print('✅ CLIP evaluation complete. Test accuracy:', round(clip_test_acc, 3))

## 7. Results Table — All 3 Approaches, All 4 Categories Combined

In [ ]:
results_df = pd.DataFrame(all_results).sort_values('test_acc', ascending=False).reset_index(drop=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
results_df.round(4)

In [ ]:
results_df.to_csv('all4_categories_model_comparison_results.csv', index=False)
print('Saved results to all4_categories_model_comparison_results.csv')

## 8. Per-Category Accuracy Breakdown

This is the key question for deciding whether to launch visual search on
all 4 categories at once: does accuracy hold up evenly, or does a thinner
category (Automobiles, only 21 items) perform noticeably worse?

In [ ]:
per_cat_df = pd.DataFrame(all_per_category_acc).round(4)
per_cat_df.index.name = 'Category'
per_cat_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
per_cat_df.plot(kind='bar', ax=ax)
ax.set_title('Test Accuracy by Category — Each Model')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.05)
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print('Item counts per category, for context on the accuracy numbers above:')
print(split_df.drop_duplicates('products_id').groupby('category').size().reindex(CATEGORIES))
print()
print('Lower accuracy in a low-count category is expected — fewer examples to')
print('learn from doesn\'t just mean less training data, it can also mean fewer')
print('confusable near-duplicates were present to stress-test the model against.')
print('Interpret a small category\'s accuracy with that in mind, not at face value.')

## 9. Weakest Items — Per Model

In [ ]:
for model_name, report in all_per_class_reports.items():
    report_df = pd.DataFrame(report).transpose()
    class_rows = report_df.iloc[:NUM_CLASSES]
    weakest = class_rows[class_rows['f1-score'] == 0]
    weakest_with_cat = [(pid, idx_to_category.get(pid, '?')) for pid in weakest.index.tolist()]
    print(f"\n--- {model_name}: {len(weakest)} items misidentified out of {NUM_CLASSES} ---")
    for pid, cat in weakest_with_cat[:30]:
        print(f"  {pid} ({cat})")

## 10. Training Curves (CNN Models Only)

In [ ]:
fig, axes = plt.subplots(len(all_histories), 2, figsize=(12, 4 * len(all_histories)))
if len(all_histories) == 1:
    axes = axes.reshape(1, -1)

for i, (model_name, history) in enumerate(all_histories.items()):
    epochs_range = range(1, len(history['train_acc']) + 1)
    best_ep = results_df[results_df['model_name'] == model_name]['best_epoch'].values[0]

    axes[i, 0].plot(epochs_range, history['train_acc'], label='Train Acc', marker='o', markersize=3)
    axes[i, 0].plot(epochs_range, history['val_acc'], label='Val Acc', marker='s', markersize=3)
    axes[i, 0].axvline(best_ep, color='green', linestyle='--', alpha=0.6, label=f'Best epoch ({best_ep})')
    axes[i, 0].set_title(f'{model_name} — Accuracy')
    axes[i, 0].legend(fontsize=8)

    axes[i, 1].plot(epochs_range, history['train_loss'], label='Train Loss', marker='o', markersize=3)
    axes[i, 1].plot(epochs_range, history['val_loss'], label='Val Loss', marker='s', markersize=3)
    axes[i, 1].axvline(best_ep, color='green', linestyle='--', alpha=0.6, label=f'Best epoch ({best_ep})')
    axes[i, 1].set_title(f'{model_name} — Loss')
    axes[i, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 11. Final Comparison — Radar Chart & Weighted Leaderboard

In [ ]:
radar_metrics = {
    'Test Accuracy': ('test_acc', False),
    'F1 (macro)': ('f1_macro', False),
    'Top-3 Accuracy': ('top3_acc', False),
    'ROC-AUC': ('roc_auc_macro', False),
    'Small Size': ('model_size_mb', True),
    'Fast Inference': ('inference_ms_per_image', True),
}

radar_df = results_df.set_index('model_name')
normalized = pd.DataFrame(index=radar_df.index)
for label, (col, invert) in radar_metrics.items():
    values = radar_df[col].fillna(radar_df[col].median())
    vmin, vmax = values.min(), values.max()
    normalized[label] = 1.0 if vmax == vmin else ((1 - (values - vmin) / (vmax - vmin)) if invert else (values - vmin) / (vmax - vmin))

labels = list(radar_metrics.keys())
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = sns.color_palette('tab10', len(normalized))
for i, (model_name, row) in enumerate(normalized.iterrows()):
    values = row.tolist(); values += values[:1]
    ax.plot(angles, values, label=model_name, color=colors[i], linewidth=2)
    ax.fill(angles, values, color=colors[i], alpha=0.08)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, fontsize=9)
ax.set_yticklabels([])
ax.set_title('All 4 Categories, Unified Model — Comparison', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
def minmax_norm(series, invert=False):
    vmin, vmax = series.min(), series.max()
    if vmax == vmin:
        return pd.Series(1.0, index=series.index)
    norm = (series - vmin) / (vmax - vmin)
    return (1 - norm) if invert else norm

score_df = results_df.copy()
score_df['norm_acc'] = minmax_norm(score_df['test_acc'])
score_df['norm_f1'] = minmax_norm(score_df['f1_macro'])
score_df['norm_size'] = minmax_norm(score_df['model_size_mb'], invert=True)
score_df['norm_speed'] = minmax_norm(score_df['inference_ms_per_image'], invert=True)
score_df['weighted_score'] = (0.50*score_df['norm_acc'] + 0.20*score_df['norm_f1'] +
                                0.15*score_df['norm_size'] + 0.15*score_df['norm_speed'])

leaderboard = score_df.sort_values('weighted_score', ascending=False)
leaderboard[['model_name', 'test_acc', 'top3_acc', 'f1_macro', 'model_size_mb',
             'inference_ms_per_image', 'weighted_score']].round(4)

In [ ]:
winner = leaderboard.iloc[0]
print('='*60)
print('RECOMMENDATION — Unified Model, All 4 Categories (838 items)')
print('='*60)
print(f"Best overall: {winner['model_name']}")
print(f"  Test accuracy: {winner['test_acc']:.3f}")
print(f"  Top-3 accuracy: {winner['top3_acc']:.3f}")
print(f"  Model size: {winner['model_size_mb']:.1f} MB")
print(f"  Inference speed: {winner['inference_ms_per_image']:.2f} ms/image")
print()
print('Check Section 8 (per-category breakdown) before finalizing — a strong')
print('overall number can still hide a weak category. If one category lags')
print('noticeably behind the others, that\'s worth addressing (more real photos')
print('for that category) before launching visual search across all 4 at once.')